# Crystal Region-Holdout Experiment

Закрывает training-side leakage в crystal-retrieval: SimCLR обучается только
на 4 из 6 азимутальных секторов полусферы, retrieval оценивается на 2 held-out.

**Шаги**:
1. Setup — mount Drive, clone/pull репо, поставить зависимости.
2. Data — распаковать `crystal_data_v1.zip`, **скопировать** `patches.npy` из Drive на локальный SSD.
3. Train — SimCLR на train-секторах (~1ч на T4).
4. Extract — эмбеддинги на ВСЕХ 150k патчах.
5. Backup — скопировать checkpoint + эмбеддинги на Drive.

Retrieval-сравнение делается локально после скачивания эмбеддингов.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess

PROJ = '/content/diploma_project'
REPO = 'https://github.com/daniilctrl/sem-image-analysis.git'

if os.path.exists(f'{PROJ}/.git'):
    # На случай, если в работочей копии остался ранее распакованный
    # split CSV (он теперь в git и блокирует pull): сбрасываем рабочую
    # директорию к origin/master.
    !cd {PROJ} && git fetch origin && git reset --hard origin/master
else:
    !git clone {REPO} {PROJ}

print('HEAD:')
!cd {PROJ} && git log --oneline -3

In [ ]:
# Зависимости (faiss-cpu требуется для retrieval, но проще поставить заодно;
# torch + sklearn в Colab уже есть)
!pip install -q faiss-cpu tqdm

## 2. Data

- `crystal_data_v1.zip` (~9 МБ) → `data/crystal/patches/patches_metadata.csv`
- split CSV приходит через `git pull` (уже под контролем git)
- `patches.npy` (~2.9 ГБ) **копируется** из Drive на локальный SSD Colab.
  Симлинком оставлять НЕЛЬЗЯ: чтение через DataLoader из Drive даёт
  ~26 сек/итерацию вместо <1 сек на T4. Копирование ~2 минуты.

In [ ]:
# Подставь свой путь, если zip лежит не там:
ZIP_DRIVE = '/content/drive/MyDrive/diploma_data/crystal_data_v1.zip'
PATCHES_DRIVE = '/content/drive/MyDrive/colab_crystal/patches.npy'

assert os.path.exists(ZIP_DRIVE), f'zip not found: {ZIP_DRIVE}'
assert os.path.exists(PATCHES_DRIVE), f'patches.npy not found: {PATCHES_DRIVE}'

# Распаковка zip (только patches_metadata.csv)
!cp {ZIP_DRIVE} /content/crystal_data_v1.zip
!unzip -qo /content/crystal_data_v1.zip -d {PROJ}/
!rm /content/crystal_data_v1.zip

# Копируем patches.npy из Drive на локальный SSD. Это критично для
# скорости тренировки: чтение из примонтированного Drive в DataLoader
# выдаёт ~26 сек/итерацию вместо <1 сек.
import shutil, time
PATCHES_LOCAL = f'{PROJ}/data/crystal/patches/patches.npy'
os.makedirs(os.path.dirname(PATCHES_LOCAL), exist_ok=True)
if os.path.lexists(PATCHES_LOCAL):
    os.remove(PATCHES_LOCAL)
t = time.time()
shutil.copy(PATCHES_DRIVE, PATCHES_LOCAL)
size_gb = os.path.getsize(PATCHES_LOCAL) / 1e9
print(f'Copied patches.npy ({size_gb:.2f} GB) in {time.time() - t:.1f}s')

# Sanity check всех трёх артефактов
for f in [
    'data/crystal/patches/patches.npy',
    'data/crystal/patches/patches_metadata.csv',
    'data/crystal/splits/region_holdout_v1.csv',
]:
    p = f'{PROJ}/{f}'
    assert os.path.exists(p), f'missing: {p}'
    assert not os.path.islink(p), f'{p} is a symlink — performance will be terrible'
    size = os.path.getsize(p)
    print(f'OK  {f:55s}  {size/1e6:>8.2f} MB')
print('\nAll three crystal artifacts in place (real files, not symlinks).')

In [ ]:
# Быстрая проверка, что split CSV выглядит правильно
import pandas as pd
df = pd.read_csv(f'{PROJ}/data/crystal/splits/region_holdout_v1.csv')
print('split sizes:', df['split'].value_counts().to_dict())
print('regions:', df['region_id'].value_counts().sort_index().to_dict())

## 3. Train SimCLR на train-секторах

In [ ]:
RUN_NAME = 'crystal_simclr_holdout_v1'
CKPT_DIR = f'/content/drive/MyDrive/diploma_checkpoints/{RUN_NAME}'

os.makedirs(CKPT_DIR, exist_ok=True)

!cd {PROJ} && python -m src.crystal.train_crystal \
    --epochs 50 \
    --batch_size 64 \
    --learning_rate 3e-4 \
    --temperature 0.2 \
    --workers 4 \
    --output_dir {CKPT_DIR} \
    --split_csv data/crystal/splits/region_holdout_v1.csv \
    --split_filter train \
    --save_every 10 \
    --seed 42

In [ ]:
# Проверим, что best-checkpoint появился
!ls -lh {CKPT_DIR}

### Resume после обрыва Colab-сессии

Если сессия отвалилась посреди тренинга, выполни ячейку ниже вместо
обычного train. Подставь `LAST_EPOCH` — номер последнего сохранённого
чекпоинта (например 30). Чекпоинты пишутся каждые `save_every=10` эпох.

In [ ]:
# RESUME ONLY — раскомментируй и заполни перед запуском
# LAST_EPOCH = 30
# TOTAL_EPOCHS = 50
# REMAINING = TOTAL_EPOCHS - LAST_EPOCH
# !cd {PROJ} && python -m src.crystal.train_crystal \
#     --epochs {REMAINING} \
#     --batch_size 64 \
#     --workers 4 \
#     --output_dir {CKPT_DIR} \
#     --split_csv data/crystal/splits/region_holdout_v1.csv \
#     --split_filter train \
#     --resume {CKPT_DIR}/crystal_simclr_epoch_{LAST_EPOCH}.pth \
#     --start_epoch {LAST_EPOCH} \
#     --seed 42

## 4. Extract эмбеддинги на ВСЕХ 150k патчах

Сама модель обучалась только на train-секторах, но эмбеддинги нужны
на полном датасете: при retrieval с `--restrict_query_split test`
queries будут только из test, а candidate-pool — из всех 150k патчей.

In [ ]:
EMB_DIR = f'/content/drive/MyDrive/diploma_embeddings/crystal_holdout'
os.makedirs(EMB_DIR, exist_ok=True)

!cd {PROJ} && python -m src.crystal.extract_crystal_embeddings \
    --checkpoint {CKPT_DIR}/crystal_simclr_best.pth \
    --output_dir {EMB_DIR} \
    --batch_size 256 \
    --workers 4 \
    --seed 42

In [ ]:
# Проверим артефакты в Drive
!ls -lh {EMB_DIR}

## 5. Скачивание эмбеддингов локально

Папка `crystal_holdout/` в Drive весит ~600 МБ (`crystal_embeddings.npy`
+ `embeddings_metadata.csv` + cluster_labels). Скачай всю папку через
Google Drive в локальный `data/crystal/embeddings_holdout/` и запусти
финальный шаг 6 локально (без GPU).

Альтернатива — сжать в zip и скачать одним файлом:

In [ ]:
# Опционально: zip эмбеддингов на Drive (одним файлом удобнее качать)
ZIP_OUT = '/content/drive/MyDrive/diploma_data/crystal_holdout_embeddings.zip'
!cd /content/drive/MyDrive/diploma_embeddings && zip -qr {ZIP_OUT} crystal_holdout
size_gb = os.path.getsize(ZIP_OUT) / 1e9
print(f'Zipped to: {ZIP_OUT} ({size_gb:.2f} GB)')

## 6. Локально (после скачивания)

```bash
# Распаковать crystal_holdout_embeddings.zip в data/crystal/embeddings_holdout/

# (a) Query=test, без spatial split — чистый holdout-эффект
python -m src.crystal.retrieve_crystal \
    --emb_path data/crystal/embeddings_holdout/crystal_embeddings.npy \
    --meta_path data/crystal/embeddings_holdout/embeddings_metadata.csv \
    --split_csv data/crystal/splits/region_holdout_v1.csv \
    --restrict_query_split test \
    --model_name simclr_holdout \
    --output_dir data/results/crystal_leakage_check

# (b) Query=test + spatial split 12° — комбо защита
python -m src.crystal.retrieve_crystal \
    --emb_path data/crystal/embeddings_holdout/crystal_embeddings.npy \
    --meta_path data/crystal/embeddings_holdout/embeddings_metadata.csv \
    --split_csv data/crystal/splits/region_holdout_v1.csv \
    --restrict_query_split test \
    --spatial_split_deg 12 \
    --model_name simclr_holdout \
    --output_dir data/results/crystal_leakage_check
```